##PORTAFOLIO POSITIVO


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.lines import Line2D

def generar_dashboard_portafolio_pmp():
    # --- 1. PROCESAMIENTO ANALÍTICO DE DATOS (Simulación de 50 Proyectos) ---
    np.random.seed(42)
    n_proyectos = 50

    # Generate CPI values to meet the desired distribution:
    # 40 Saludables (CPI >= 1.00)
    # 5 Observación (0.90 <= CPI < 1.00)
    # 5 Crítico (CPI < 0.90)
    cpi_saludable = np.random.uniform(1.00, 1.20, 40).round(2)
    cpi_observacion = np.random.uniform(0.90, 0.99, 5).round(2)
    cpi_critical = np.random.uniform(0.70, 0.89, 5).round(2)

    # Combine and shuffle CPIs to ensure random assignment to projects
    all_cpi = np.concatenate([cpi_saludable, cpi_observacion, cpi_critical])
    np.random.shuffle(all_cpi)

    df = pd.DataFrame({
        'ID_Display': [f"P{i+1}" for i in range(n_proyectos)],
        'BAC': np.random.uniform(500000, 2000000, n_proyectos).round(0),
        'CPI': all_cpi, # Assign the new CPIs
        'SPI': np.random.normal(0.94, 0.10, n_proyectos).round(2)
    })

    # Cálculos bajo estándares EVM (Earned Value Management)
    df['EAC'] = (df['BAC'] / df['CPI']).round(0)
    df['VAC'] = df['BAC'] - df['EAC']
    df['EV'] = df['BAC'] * 0.5
    df['AC'] = df['EV'] / df['CPI']

    # Lógica de Semáforos simplificada
    def get_status(cpi):
        if cpi < 0.90: return 'Crítico'
        if cpi < 1.00: return 'Observación'
        return 'Saludable'

    df['Status'] = df['CPI'].apply(get_status)

    # Métricas Globales para el Informe Dinámico
    stats_counts = df['Status'].value_counts().reindex(['Crítico', 'Observación', 'Saludable'], fill_value=0)
    tot_bac = df['BAC'].sum()
    tot_eac = df['EAC'].sum()
    tot_vac = df['VAC'].sum()
    global_cpi = df['CPI'].mean()
    proyecto_critico = df.loc[df['VAC'].idxmin()]

    # --- 2. CONFIGURACIÓN VISUAL E IDENTIDAD CORPORATIVA ---
    c_header_bg = '#003366'    # Azul Oscuro Institucional
    c_gris_claro = '#D5D8DC'   # Gris Claro para tipografía en franjas
    c_critico = '#E74C3C'      # Rojo
    c_observacion = '#F1C40F'  # Amarillo
    c_saludable = '#27AE60'    # Verde

    status_colors = {'Crítico': c_critico, 'Observación': c_observacion, 'Saludable': c_saludable}

    # --- 3. CONSTRUCCIÓN DEL DASHBOARD (Formato Carta Horizontal) ---
    fig = plt.figure(figsize=(19, 11))
    gs = gridspec.GridSpec(4, 3,
                           height_ratios=[0.08, 0.42, 0.44, 0.06],
                           width_ratios=[0.35, 0.35, 0.30])

    # Desplazamiento a la derecha para alinear ejes con el inicio de las franjas
    plt.subplots_adjust(left=0.08, right=0.97, top=0.96, bottom=0.04, wspace=0.18, hspace=0.6026)

    # --- A. ENCABEZADO (HEADER) ---
    # Reemplazado con add_axes para abarcar todo el ancho de la figura
    ax_header = fig.add_axes([0, 0.88, 1, 0.08]) # [left, bottom, width, height] en coordenadas de figura
    ax_header.axis('off')
    ax_header.add_patch(patches.Rectangle((0, 0), 1, 1, transform=ax_header.transAxes, color=c_header_bg))
    # Ajuste de posición y alineación del texto del encabezado
    ax_header.text(0.02, 0.5, "DASHBOARD EJECUTIVO DE PORTAFOLIO | ANALÍTICA PREDICTIVA",
                   color=c_gris_claro, fontsize=26, fontweight='bold', va='center', ha='left')

    # --- B. MAPA DE SALUD (Top Left) ---
    ax_scatter = fig.add_subplot(gs[1, 0])
    for status, color in status_colors.items():
        subset = df[df['Status'] == status]
        ax_scatter.scatter(subset['SPI'], subset['CPI'], s=subset['BAC']/7500, c=color, alpha=0.75, edgecolors='black')

    ax_scatter.axhline(1.0, color='#34495E', linestyle='--', linewidth=0.8)
    ax_scatter.axvline(1.0, color='#34495E', linestyle='--', linewidth=0.8)
    ax_scatter.set_title("MAPA DE SALUD (CPI vs SPI)", fontsize=16, fontweight='normal', pad=15)
    ax_scatter.set_xlabel("Eficiencia Cronograma (SPI)")
    ax_scatter.set_ylabel("Eficiencia Costos (CPI)")

    # Convenciones simplificadas (Solo círculos)
    legend_elements = [Line2D([0], [0], marker='o', color='w', label='Crítico', markerfacecolor=c_critico, markersize=12),
                       Line2D([0], [0], marker='o', color='w', label='Observación', markerfacecolor=c_observacion, markersize=12),
                       Line2D([0], [0], marker='o', color='w', label='Saludable', markerfacecolor=c_saludable, markersize=12)]
    ax_scatter.legend(handles=legend_elements, loc='lower left', frameon=False, fontsize=10)

    # --- C. DISTRIBUCIÓN DE ESTADOS (Top Middle) ---
    ax_pie = fig.add_subplot(gs[1, 1])
    def func_pct(pct, allvals):
        absolute = int(round(pct/100.*np.sum(allvals)))
        return f"{pct:.1f}%\n({absolute} Proy.)"

    ax_pie.pie(stats_counts.values, labels=stats_counts.index,
               autopct=lambda pct: func_pct(pct, stats_counts.values),
               startangle=140, colors=[c_critico, c_observacion, c_saludable],
               wedgeprops={'edgecolor': 'white', 'linewidth': 2}, pctdistance=0.7)
    ax_pie.set_title("DISTRIBUCIÓN DE ESTADOS", fontsize=16, fontweight='normal', pad=15)

    # --- D. VARIACIÓN A LA CONCLUSIÓN (Bottom - Ampliado Horizontalmente) ---
    ax_bar = fig.add_subplot(gs[2, 0:2])
    bar_colors = [status_colors[s] for s in df['Status']]
    ax_bar.bar(df['ID_Display'], df['VAC'], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax_bar.set_title("VARIACIÓN A LA CONCLUSIÓN (VAC)", fontsize=16, fontweight='normal', pad=15)
    ax_bar.set_xticks(range(len(df['ID_Display'])))
    ax_bar.set_xticklabels(df['ID_Display'], rotation=90, fontsize=9)
    ax_bar.set_ylabel("Variación ($)")

    # --- E. INFORME TÉCNICO DINÁMICO (Lado Derecho) ---
    ax_text = fig.add_subplot(gs[1:3, 2])
    ax_text.axis('off')

    txt_report = (
        f"INFORME EJECUTIVO FINAL - PORTAFOLIO 2026\n"
        f"=====================================================\n\n"
        f"1. RESULTADOS GLOBALES:\n"
        f"----------------------\n"
        f"  - Presupuesto Original (BAC): ${tot_bac:,.0f}\n"
        f"  - Estimado al Cierre (EAC): ${tot_eac:,.0f}\n"
        f"  - Variación Final (VAC): ${tot_vac:,.0f}\n"
        f"  - Índice Eficiencia Costo: {global_cpi:.2f}\n\n"
        f"2. ANÁLISIS DE CAUSA RAÍZ (PUNTO CRÍTICO):\n"
        f"------------------------------------------\n"
        f"  El proyecto '{proyecto_critico['ID_Display']}' presenta la mayor\n"
        f"  desviación negativa (${proyecto_critico['VAC']:,.0f}).\n\n"
        f"  Hallazgos:\n"
        f"  - El CPI de {proyecto_critico['CPI']:.2f} indica que por cada\n"
        f"    dólar invertido, solo se genera ${proyecto_critico['CPI']:.2f}.\n"
        f"  - Tendencia: El portafolio muestra una\n"
        f"    desviación acumulada en el {((stats_counts['Crítico']/n_proyectos)*100):.0f}% de la cartera.\n\n"
        f"3. RECOMENDACIONES ESTRATÉGICAS:\n"
        f"--------------------------------\n"
        f"  A. ACCIÓN INMEDIATA (CORTO PLAZO):\n"
        f"    - Auditoría técnica en {proyecto_critico['ID_Display']}.\n"
        f"    - Congelar cambios de alcance.\n\n"
        f"  B. GESTIÓN CONTRACTUAL:\n"
        f"    - Revisar cláusulas de penalización.\n\n"
        f"  C. PROYECCIÓN:\n"
        f"    - De mantenerse la tendencia, se requerirá\n"
        f"      una inyección de capital de aprox. ${abs(tot_vac):,.0f}."
    )

    ax_text.text(0.05, 1.0, txt_report, fontsize=11, family='monospace',
                 va='top', linespacing=1.4, color=c_header_bg)

    # --- F. PIE DE PÁGINA (FOOTER) ---
    # Reemplazado con add_axes para abarcar todo el ancho de la figura
    ax_foot = fig.add_axes([0, 0.00, 1, 0.06]) # Ajustado ligeramente 'bottom' para mantener coherencia si no hay margen
    ax_foot.axis('off')
    ax_foot.add_patch(patches.Rectangle((0, 0), 1, 1, transform=ax_foot.transAxes, color=c_header_bg))

    ax_foot.text(0.02, 0.5, "EMILIO PALACÍN GÓMEZ", color=c_gris_claro, fontsize=14, fontweight='bold', va='center')
    ax_foot.text(0.5, 0.5, "Consultor PMO | Project Manager PMP® | BI, Reporting Ejecutivo y Control de Proyectos",
                 color=c_gris_claro, fontsize=11, va='center', ha='center')
    ax_foot.text(0.98, 0.5, "www.linkedin.com/in/emiliopalacin", color=c_gris_claro, fontsize=11, va='center', ha='right')

    plt.savefig("Dashboard_Portafolio_Positivo.png", dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    generar_dashboard_portafolio_pmp()

##PORTAFOLIO EN CRISIS


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.lines import Line2D

def generar_dashboard_portafolio_pmp():
    # --- 1. PROCESAMIENTO ANALÍTICO DE DATOS (Simulación de 50 Proyectos) ---
    np.random.seed(42)
    n_proyectos = 50
    df = pd.DataFrame({
        'ID_Display': [f"P{i+1}" for i in range(n_proyectos)],
        'BAC': np.random.uniform(500000, 2000000, n_proyectos).round(0),
        'CPI': np.random.normal(0.96, 0.12, n_proyectos).round(2),
        'SPI': np.random.normal(0.94, 0.10, n_proyectos).round(2)
    })

    # Cálculos bajo estándares EVM (Earned Value Management)
    df['EAC'] = (df['BAC'] / df['CPI']).round(0)
    df['VAC'] = df['BAC'] - df['EAC']
    df['EV'] = df['BAC'] * 0.5
    df['AC'] = df['EV'] / df['CPI']

    # Lógica de Semáforos simplificada
    def get_status(cpi):
        if cpi < 0.90: return 'Crítico'
        if cpi < 1.00: return 'Observación'
        return 'Saludable'

    df['Status'] = df['CPI'].apply(get_status)

    # Métricas Globales para el Informe Dinámico
    stats_counts = df['Status'].value_counts().reindex(['Crítico', 'Observación', 'Saludable'], fill_value=0)
    tot_bac = df['BAC'].sum()
    tot_eac = df['EAC'].sum()
    tot_vac = df['VAC'].sum()
    global_cpi = df['CPI'].mean()
    proyecto_critico = df.loc[df['VAC'].idxmin()]

    # --- 2. CONFIGURACIÓN VISUAL E IDENTIDAD CORPORATIVA ---
    c_header_bg = '#003366'    # Azul Oscuro Institucional
    c_gris_claro = '#D5D8DC'   # Gris Claro para tipografía en franjas
    c_critico = '#E74C3C'      # Rojo
    c_observacion = '#F1C40F'  # Amarillo
    c_saludable = '#27AE60'    # Verde

    status_colors = {'Crítico': c_critico, 'Observación': c_observacion, 'Saludable': c_saludable}

    # --- 3. CONSTRUCCIÓN DEL DASHBOARD (Formato Carta Horizontal) ---
    fig = plt.figure(figsize=(20, 11))
    gs = gridspec.GridSpec(4, 3,
                           height_ratios=[0.08, 0.42, 0.44, 0.06],
                           width_ratios=[0.35, 0.35, 0.30])

    # Desplazamiento a la derecha para alinear ejes con el inicio de las franjas
    plt.subplots_adjust(left=0.08, right=0.97, top=0.96, bottom=0.04, wspace=0.18, hspace=0.6026)

    # --- A. ENCABEZADO (HEADER) ---
    # Reemplazado con add_axes para abarcar todo el ancho de la figura
    ax_header = fig.add_axes([0, 0.88, 1, 0.08]) # [left, bottom, width, height] en coordenadas de figura
    ax_header.axis('off')
    ax_header.add_patch(patches.Rectangle((0, 0), 1, 1, transform=ax_header.transAxes, color=c_header_bg))
    # Ajuste de posición y alineación del texto del encabezado
    ax_header.text(0.02, 0.5, "DASHBOARD EJECUTIVO DE PORTAFOLIO | ANALÍTICA PREDICTIVA",
                   color=c_gris_claro, fontsize=26, fontweight='bold', va='center', ha='left')

    # --- B. MAPA DE SALUD (Top Left) ---
    ax_scatter = fig.add_subplot(gs[1, 0])
    for status, color in status_colors.items():
        subset = df[df['Status'] == status]
        ax_scatter.scatter(subset['SPI'], subset['CPI'], s=subset['BAC']/7500, c=color, alpha=0.75, edgecolors='black')

    ax_scatter.axhline(1.0, color='#34495E', linestyle='--', linewidth=0.8)
    ax_scatter.axvline(1.0, color='#34495E', linestyle='--', linewidth=0.8)
    ax_scatter.set_title("MAPA DE SALUD (CPI vs SPI)", fontsize=16, fontweight='normal', pad=15)
    ax_scatter.set_xlabel("Eficiencia Cronograma (SPI)")
    ax_scatter.set_ylabel("Eficiencia Costos (CPI)")

    # Convenciones simplificadas (Solo círculos)
    legend_elements = [Line2D([0], [0], marker='o', color='w', label='Crítico', markerfacecolor=c_critico, markersize=12),
                       Line2D([0], [0], marker='o', color='w', label='Observación', markerfacecolor=c_observacion, markersize=12),
                       Line2D([0], [0], marker='o', color='w', label='Saludable', markerfacecolor=c_saludable, markersize=12)]
    ax_scatter.legend(handles=legend_elements, loc='lower left', frameon=False, fontsize=10)

    # --- C. DISTRIBUCIÓN DE ESTADOS (Top Middle) ---
    ax_pie = fig.add_subplot(gs[1, 1])
    def func_pct(pct, allvals):
        absolute = int(round(pct/100.*np.sum(allvals)))
        return f"{pct:.1f}%\n({absolute} Proy.)"

    ax_pie.pie(stats_counts.values, labels=stats_counts.index,
               autopct=lambda pct: func_pct(pct, stats_counts.values),
               startangle=140, colors=[c_critico, c_observacion, c_saludable],
               wedgeprops={'edgecolor': 'white', 'linewidth': 2}, pctdistance=0.7)
    ax_pie.set_title("DISTRIBUCIÓN DE ESTADOS", fontsize=16, fontweight='normal', pad=15)

    # --- D. VARIACIÓN A LA CONCLUSIÓN (Bottom - Ampliado Horizontalmente) ---
    ax_bar = fig.add_subplot(gs[2, 0:2])
    bar_colors = [status_colors[s] for s in df['Status']]
    ax_bar.bar(df['ID_Display'], df['VAC'], color=bar_colors, edgecolor='black', linewidth=0.5)
    ax_bar.set_title("VARIACIÓN A LA CONCLUSIÓN (VAC)", fontsize=16, fontweight='normal', pad=15)
    ax_bar.set_xticks(range(len(df['ID_Display'])))
    ax_bar.set_xticklabels(df['ID_Display'], rotation=90, fontsize=9)
    ax_bar.set_ylabel("Variación ($)")

    # --- E. INFORME TÉCNICO DINÁMICO (Lado Derecho) ---
    ax_text = fig.add_subplot(gs[1:3, 2])
    ax_text.axis('off')

    txt_report = (
        f"INFORME EJECUTIVO FINAL - PORTAFOLIO 2026\n"
        f"=====================================================\n\n"
        f"1. RESULTADOS GLOBALES:\n"
        f"----------------------\n"
        f"  - Presupuesto Original (BAC): ${tot_bac:,.0f}\n"
        f"  - Estimado al Cierre (EAC): ${tot_eac:,.0f}\n"
        f"  - Variación Final (VAC): ${tot_vac:,.0f}\n"
        f"  - Índice Eficiencia Costo: {global_cpi:.2f}\n\n"
        f"2. ANÁLISIS DE CAUSA RAÍZ (PUNTO CRÍTICO):\n"
        f"------------------------------------------\n"
        f"  El proyecto '{proyecto_critico['ID_Display']}' presenta la mayor\n"
        f"  desviación negativa (${proyecto_critico['VAC']:,.0f}).\n\n"
        f"  Hallazgos:\n"
        f"  - El CPI de {proyecto_critico['CPI']:.2f} indica que por cada\n"
        f"    dólar invertido, solo se genera ${proyecto_critico['CPI']:.2f}.\n"
        f"  - Tendencia: El portafolio muestra una\n"
        f"    desviación acumulada en el {((stats_counts['Crítico']/n_proyectos)*100):.0f}% de la cartera.\n\n"
        f"3. RECOMENDACIONES ESTRATÉGICAS:\n"
        f"--------------------------------\n"
        f"  A. ACCIÓN INMEDIATA (CORTO PLAZO):\n"
        f"    - Auditoría técnica en {proyecto_critico['ID_Display']}.\n"
        f"    - Congelar cambios de alcance.\n\n"
        f"  B. GESTIÓN CONTRACTUAL:\n"
        f"    - Revisar cláusulas de penalización.\n\n"
        f"  C. PROYECCIÓN:\n"
        f"    - De mantenerse la tendencia, se requerirá\n"
        f"      una inyección de capital de aprox. ${abs(tot_vac):,.0f}."
    )

    ax_text.text(0.05, 1.0, txt_report, fontsize=11, family='monospace',
                 va='top', linespacing=1.4, color=c_header_bg)

    # --- F. PIE DE PÁGINA (FOOTER) ---
    # Reemplazado con add_axes para abarcar todo el ancho de la figura
    ax_foot = fig.add_axes([0, 0.00, 1, 0.06]) # Ajustado ligeramente 'bottom' para mantener coherencia si no hay margen
    ax_foot.axis('off')
    ax_foot.add_patch(patches.Rectangle((0, 0), 1, 1, transform=ax_foot.transAxes, color=c_header_bg))

    ax_foot.text(0.02, 0.5, "EMILIO PALACÍN GÓMEZ", color=c_gris_claro, fontsize=14, fontweight='bold', va='center')
    ax_foot.text(0.5, 0.5, "Consultor PMO | Project Manager PMP® | BI, Reporting Ejecutivo y Control de Proyectos",
                 color=c_gris_claro, fontsize=11, va='center', ha='center')
    ax_foot.text(0.98, 0.5, "www.linkedin.com/in/emiliopalacin", color=c_gris_claro, fontsize=11, va='center', ha='right')

    plt.savefig("Dashboard_Portafolio_Crisis.png", dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    generar_dashboard_portafolio_pmp()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.lines import Line2D

# Configuración de Colores Institucionales
c_header_bg = '#003366'    # Azul Oscuro Institucional
c_gris_claro = '#D5D8DC'   # Gris Claro
status_colors = {'Crítico': '#E74C3C', 'Observación': '#F1C40F', 'Saludable': '#27AE60'}

def generar_datos(seed, modo='exitoso'):
    np.random.seed(seed)
    n = 50
    if modo == 'exitoso':
        cpis = np.concatenate([np.random.uniform(1.02, 1.15, 40), np.random.uniform(0.92, 0.99, 5), np.random.uniform(0.80, 0.89, 5)])
        spis = np.random.normal(1.02, 0.05, n)
    else:
        cpis = np.concatenate([np.random.uniform(1.00, 1.05, 10), np.random.uniform(0.90, 0.99, 15), np.random.uniform(0.70, 0.89, 25)])
        spis = np.random.normal(0.88, 0.10, n)
    np.random.shuffle(cpis)
    df = pd.DataFrame({'ID': [f"P{i+1}" for i in range(n)], 'BAC': np.random.uniform(800000, 2500000, n), 'CPI': cpis.round(2), 'SPI': spis.round(2)})
    df['Status'] = df['CPI'].apply(lambda x: 'Crítico' if x < 0.9 else ('Observación' if x < 1.0 else 'Saludable'))
    return df

def plot_dashboard():
    df_exito = generar_datos(42, 'exitoso')
    df_crisis = generar_datos(99, 'crisis')

    fig = plt.figure(figsize=(22, 12))
    # Ajuste para franjas de lado a lado
    ax_h = fig.add_axes([0, 0.92, 1, 0.08])
    ax_h.axis('off')
    ax_h.add_patch(patches.Rectangle((0, 0), 1, 1, color=c_header_bg))
    ax_h.text(0.02, 0.5, "COMPARATIVA ESTRATÉGICA DE PORTAFOLIOS | DASHBOARD DE GOBERNANZA", color=c_gris_claro, fontsize=24, fontweight='bold', va='center')

    ax_f = fig.add_axes([0, 0, 1, 0.06])
    ax_f.axis('off')
    ax_f.add_patch(patches.Rectangle((0, 0), 1, 1, color=c_header_bg))
    ax_f.text(0.02, 0.5, "EMILIO PALACÍN GÓMEZ", color=c_gris_claro, fontsize=14, fontweight='bold', va='center')
    ax_f.text(0.98, 0.5, "www.linkedin.com/in/emiliopalacin", color=c_gris_claro, fontsize=11, va='center', ha='right')

    # Distribución equilibrada entre las franjas
    gs = gridspec.GridSpec(2, 2, figure=fig, left=0.1, right=0.9, top=0.88, bottom=0.1, hspace=0.4, wspace=0.3)

    for i, (df, title) in enumerate([(df_exito, "PORTAFOLIO A (EXITOSO)"), (df_crisis, "PORTAFOLIO B (EN CRISIS)")]):
        # Scatter
        ax_s = fig.add_subplot(gs[i, 0])
        for st, col in status_colors.items():
            sub = df[df['Status'] == st]
            ax_s.scatter(sub['SPI'], sub['CPI'], s=sub['BAC']/10000, c=col, alpha=0.6, edgecolors='black')
        ax_s.set_title(title, fontweight='bold', pad=15)
        ax_s.axhline(1, color='gray', linestyle='--'); ax_s.axvline(1, color='gray', linestyle='--')

        # Pie
        ax_p = fig.add_subplot(gs[i, 1])
        cnts = df['Status'].value_counts().reindex(['Crítico', 'Observación', 'Saludable'], fill_value=0)
        ax_p.pie(cnts, labels=cnts.index, autopct='%1.0f%%', colors=[status_colors[k] for k in cnts.index], wedgeprops={'edgecolor': 'white'})

    plt.savefig("Dashboard_Comparativo.png", dpi=300)
    plt.show()

plot_dashboard()